# Phase 2 — Pretraining on Kaggle

Trains one ~24.1M-parameter decoder-only Transformer from scratch. Run this
notebook **once per language** (`hindi`, then `assamese`) — the two models share
no data, tokenizer, vocabulary or weights.

## One-time setup

Two private Kaggle Datasets are needed. The repo is private, so the code is
uploaded rather than cloned.

1. **`lma-tokens`** — the token arrays. Locally run
   `python -m common.tokenization.tokenize_corpus --lang hindi` and `--lang assamese`, then
   upload `data/tokenized/` (~2.3 GB of `uint16` ids). The 6.4 GB text corpus is
   *not* needed here.
2. **`lma-code`** — the code. Locally run `python kaggle/stage_code_payload.py`
   and upload `kaggle/code_payload/` (~1.6 MB).
3. **Attach both** via *Add Data*.
4. **GPU on**: *Settings → Accelerator → GPU T4 x2 or P100*.
5. **Internet** — optional. Pretraining needs no network: the token arrays are
   already on disk. Internet only lets the setup cell fetch `sacrebleu`, which
   adds BLEU and chrF to the evaluation; every other metric is computed without
   it. Turn it on under *Settings → Internet* if you want those two numbers
   (Kaggle requires a verified phone number to enable it).

## Resuming after a session ends

Kaggle caps sessions at ~12 h and can evict earlier, so the run is built to be
interrupted. When a session ends:

1. *Save Version → Save & Run All (Commit)* so `/kaggle/working` is preserved.
2. In the next session, *Add Data → Notebook Output* and attach the previous
   version.
3. Set `RESUME_FROM` below to that path. Training continues from the exact
   step, optimizer and scheduler state — it does not restart.

In [ ]:
# ---- Run configuration -------------------------------------------------
LANG        = "hindi"        # "hindi" or "assamese" -- run this notebook once per language
MAX_STEPS   = 6_000          # ~590M tokens = one epoch over the corpus.
                             # Spec 2.2 targets ~500M tokens; compute-optimal for a
                             # 24.1M-param model is ~482M (20 tokens/param), so one
                             # epoch sits just above optimal. 20k steps would be 3.3
                             # epochs and ~4x the GPU time for little gain.
BATCH_SIZE  = 24             # micro-batch that sits on the GPU
GRAD_ACCUM  = 8              # effective batch = 24 x 8 x 512 = 98,304 tokens/step
LEARNING_RATE = 3e-4
WARMUP      = 500
EVAL_EVERY  = 500
CKPT_EVERY  = 500

DATA_ROOT   = "/kaggle/input/lma-tokens"   # the attached tokenized dataset;
                             # the next-but-one cell resolves the exact subdirectory
WORK        = "/kaggle/working"

# Set to a previous notebook-output path to continue a run, e.g.
#   RESUME_FROM = "/kaggle/input/phase2-hindi-v1/hindi/model"
# Leave as None for a fresh start.
RESUME_FROM = None

# Delete this language's checkpoints in /kaggle/working before training.
# /kaggle/working persists across "Run All" within a session, so a short trial
# run leaves a last.pt behind that the next run silently continues from -- which
# reports 0.0 minutes and looks like success while doing nothing. Set True when
# you want a genuinely fresh run (e.g. re-timing after a code change). Ignored
# when RESUME_FROM is set, since resuming is then the explicit intent.
FRESH_START = False

In [ ]:
# ---- Code + dependencies ----------------------------------------------
# The GitHub Classroom repo is PRIVATE, so `git clone` cannot work here: a
# Kaggle kernel has no credentials. The code is shipped as a small (~1.7 MB)
# Kaggle Dataset instead -- see kaggle/stage_code_payload.py. This also avoids
# putting a personal access token in a notebook, which would leak on sharing.
import os, sys, shutil, subprocess, pathlib

SRC = "/kaggle/working/repo"

# Find the payload by its contents rather than by an exact path: Kaggle derives
# the mount name from the dataset title, and may or may not keep the folder that
# was dragged in. Anything containing common/modeling/model.py is the payload.
_marker = "common/modeling/model.py"
# Shallowest match wins: if a payload is nested inside a wrapper folder there
# is only one hit anyway, and if two code datasets are attached the top-level
# one is the sane default rather than whichever sorts first alphabetically.
_hits = sorted(pathlib.Path("/kaggle/input").glob(f"*/**/{_marker}"),
               key=lambda q: len(q.parts))
if not _hits:
    print("Attached datasets:", sorted(os.listdir("/kaggle/input")) or "(none)")
    raise SystemExit(
        "Could not find the code payload under /kaggle/input. Create a private "
        "Kaggle Dataset from kaggle/code_payload/, name it `lma-code`, and "
        "attach it to this notebook via Add Data.")
CODE_DIR = str(_hits[0].parents[2])      # .../common/modeling/model.py -> root
print("code payload found at", CODE_DIR)

# Copy out of the read-only input mount: training writes checkpoints and logs
# relative to the working directory, and /kaggle/input cannot be written to.
if os.path.exists(SRC):
    shutil.rmtree(SRC)
shutil.copytree(CODE_DIR, SRC)
os.chdir(SRC)
sys.path.insert(0, SRC)
print("code staged at", SRC)

# Evict any `common.*` already imported by an earlier run of this cell. Python
# caches modules in sys.modules, so copying fresh files over the old ones is not
# enough: `from common.modeling.train import Trainer` would keep handing back the
# object built from the previous session's code. That failure is near-invisible,
# because this cell reads the version stamp off disk and reports the NEW build
# while training silently runs the OLD one. Dropping them forces a real reimport.
_stale = [m for m in sys.modules if m == "common" or m.startswith("common.")]
for _m in _stale:
    del sys.modules[_m]
if _stale:
    print(f"dropped {len(_stale)} cached common.* modules so the new code loads")

# torch ships with the Kaggle image, and usually sentencepiece does too.
# Install only what is genuinely absent, and never let this step abort the run:
# a new Kaggle notebook has Internet OFF by default (turning it on needs phone
# verification), and pretraining itself needs no network at all -- the token
# arrays are already on disk. Failing here would block hours of GPU work over
# a metric that is optional by design.
import importlib.util

def _have(mod):
    return importlib.util.find_spec(mod) is not None

_missing = [m for m in ("sentencepiece", "sacrebleu") if not _have(m)]
if _missing:
    print("installing:", ", ".join(_missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing])
    importlib.invalidate_caches()

# Report what is actually available, and what each absence costs.
if not _have("sentencepiece"):
    print("NOTE: sentencepiece unavailable -- training and loss curves are "
          "unaffected, but the evaluation and attention cells need it to "
          "decode text. Turn Internet ON in Settings and rerun this cell.")
if not _have("sacrebleu"):
    print("NOTE: sacrebleu unavailable -- BLEU and chrF will be omitted. "
          "PPL, BPB, ROUGE-L, Distinct-n and repetition rate are computed "
          "without it, so evaluation still runs.")

# Which build of the code is actually running. A Kaggle Dataset is a snapshot:
# a repo fix reaches this notebook only after the dataset gets a new version AND
# the notebook is switched to it, so print the stamp rather than assume.
_ver = pathlib.Path(SRC) / "PAYLOAD_VERSION.txt"
print("payload version:", _ver.read_text().strip().replace("\n", " | ")
      if _ver.exists() else "UNSTAMPED -- predates versioning, almost certainly stale")

import torch
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    cap = torch.cuda.get_device_capability()
    # Report the dtype training will actually pick, and why. bf16 is only the
    # faster choice from Ampere (sm_80) on; Turing reports bf16 as supported
    # but has fp16 tensor cores only, so it runs bf16 emulated and slow.
    dt = "bfloat16" if cap[0] >= 8 else "float16 (+GradScaler)"
    print(f"GPU: {p.name}  {p.total_memory/1e9:.1f} GB  sm_{cap[0]}{cap[1]}")
    print(f"training will use: {dt}")
else:
    print("WARNING: no GPU -- Settings > Accelerator > GPU")

In [ ]:
# ---- Locate and check the tokenized data --------------------------------
# Kaggle's uploader sometimes keeps the folder you dragged, so the arrays may
# sit at <mount>/hindi_train.bin or at <mount>/tokenized/hindi_train.bin
# depending on how the dataset was created. Rather than making that a run that
# dies on cell 4, find the directory that actually holds the arrays.
# Search all of /kaggle/input, not just DATA_ROOT: Kaggle derives the mount name
# from the dataset title and will silently disambiguate a clash (`lma-tokens-1`),
# so the expected path is a hint, not a guarantee. Preferring DATA_ROOT when it
# does match keeps behaviour predictable if several copies are attached.
import pathlib

_hits = sorted(pathlib.Path("/kaggle/input").glob(f"*/**/{LANG}_train.bin"),
               key=lambda q: (not str(q).startswith(DATA_ROOT), len(q.parts)))
if not _hits:
    print("Nothing matched. Here is what is actually attached:")
    root = pathlib.Path("/kaggle/input")
    entries = sorted(root.iterdir()) if root.exists() else []
    if not entries:
        print("  (no datasets attached at all)")
    for d in entries:
        inner = sorted(p.name for p in d.iterdir())[:8]
        print(f"  {d.name}/  ->  {inner if inner else '(empty)'}")
    raise SystemExit(
        f"No {LANG}_train.bin found under /kaggle/input. Check the listing "
        f"above: if the dataset is missing, attach it via Add Data; if it is "
        f"there but empty, the upload is still processing -- wait for Kaggle "
        f"to finish and restart the session.")

DATA_DIR = str(_hits[0].parent)
print("DATA_DIR =", DATA_DIR)
if not DATA_DIR.startswith(DATA_ROOT):
    print(f"(note: found outside {DATA_ROOT} -- using the location above)")

from common.modeling.data import TokenArray

for split in ("train", "val"):
    a = TokenArray(f"{DATA_DIR}/{LANG}_{split}.bin")
    print(f"{LANG}/{split}: {len(a):,} tokens | vocab {a.vocab_size} "
          f"| {a.path.stat().st_size/1e9:.2f} GB")
    # TokenArray already cross-checks length against the sidecar JSON, so a
    # truncated upload fails here rather than corrupting training silently.

In [ ]:
# ---- Prove the implementation is correct, on this machine ---------------
# Spec 2.1 requires empirically showing the model cannot attend to the future.
# Running it here (not just locally) also catches any GPU/dtype-specific issue.
import subprocess, sys
subprocess.run([sys.executable, "-m", "common.modeling.verify_model"], check=True)

In [ ]:
# ---- Restore a previous run's checkpoints, if resuming -------------------
import shutil, pathlib

out_dir = pathlib.Path(WORK) / LANG / "model"

if FRESH_START and not RESUME_FROM and out_dir.exists():
    shutil.rmtree(out_dir)
    print(f"FRESH_START: cleared {out_dir}")

out_dir.mkdir(parents=True, exist_ok=True)

if RESUME_FROM:
    src = pathlib.Path(RESUME_FROM)
    for name in ("last.pt", "best.pt", "train_log.jsonl"):
        s = src / name
        if s.exists():
            shutil.copy2(s, out_dir / name)
            print("restored", name, f"({s.stat().st_size/1e6:.1f} MB)")
    if not (out_dir / "last.pt").exists():
        print("WARNING: no last.pt found at RESUME_FROM -- will start fresh")
elif (out_dir / "last.pt").exists():
    # RESUME_FROM is None, so a fresh run was intended -- but /kaggle/working
    # survives a "Run All" within the same session, and Trainer.maybe_resume()
    # will pick this up. If its step is already >= MAX_STEPS the training cell
    # then does nothing at all and reports 0.0 minutes, which reads like a
    # success. Say so plainly instead.
    import torch
    _step = torch.load(out_dir / "last.pt", map_location="cpu",
                       weights_only=False).get("step", "?")
    print(f"NOTE: an existing last.pt at step {_step} is in {out_dir}.")
    print("      Training will CONTINUE from it, not restart.")
    if isinstance(_step, int) and _step >= MAX_STEPS:
        print(f"      It is already at/past MAX_STEPS={MAX_STEPS}, so the next "
              f"cell will do nothing.")
        print(f"      To start over instead, run:  "
              f"import shutil; shutil.rmtree('{out_dir}')")
else:
    print("fresh run (RESUME_FROM is None)")

In [ ]:
# ---- Train -------------------------------------------------------------
# Trainer.maybe_resume() picks up last.pt automatically if the cell above
# restored one, continuing from the exact step/optimizer/scheduler state.
from common.modeling.model import ModelConfig
from common.modeling.train import TrainConfig, Trainer
import json, pathlib

mcfg = ModelConfig.load(f"{LANG}/configs/model.json")
tcfg = TrainConfig(
    language=LANG, data_dir=DATA_DIR, out_dir=str(out_dir),
    max_steps=MAX_STEPS, batch_size=BATCH_SIZE, grad_accum_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE, min_lr=LEARNING_RATE / 10, warmup_steps=WARMUP,
    eval_interval=EVAL_EVERY, checkpoint_interval=CKPT_EVERY,
)

trainer = Trainer(mcfg, tcfg)
rep = trainer.model.parameter_report()
print(f"parameters: {rep['total_trainable']:,} "
      f"(embeddings {rep['embedding_share_of_total']:.1%} of total)")
pathlib.Path(out_dir / "parameter_report.json").write_text(json.dumps(rep, indent=2))

# Hyperparameters as plain JSON beside the checkpoints. They are already inside
# the .pt payload, but a grader reading the repo should not have to unpickle a
# 290 MB tensor file to see how the run was configured -- and the spec asks for
# a configuration file alongside each checkpoint.
from dataclasses import asdict
pathlib.Path(out_dir / "config.json").write_text(json.dumps(
    {"model": asdict(mcfg), "train": asdict(tcfg)}, indent=2))

trainer.maybe_resume()
trainer.train()

In [ ]:
# ---- Loss curves (title, axis labels and legend are required) -----------
import json, pathlib
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

recs = [json.loads(l) for l in (out_dir / "train_log.jsonl").read_text().splitlines()]
tr = [(r["step"], r["train_loss"]) for r in recs if "train_loss" in r]
va = [(r["step"], r["val_loss"])   for r in recs if "val_loss"   in r]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(*zip(*tr), label="Training loss", linewidth=1.0, alpha=0.8)
if va:
    ax.plot(*zip(*va), label="Validation loss", marker="o", markersize=3, linewidth=1.5)
ax.set_title(f"{LANG.title()} — pretraining loss")
ax.set_xlabel("Optimizer step")
ax.set_ylabel("Cross-entropy loss (nats/token)")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(out_dir / f"loss_curve_{LANG}.png", dpi=150)
print("saved", out_dir / f"loss_curve_{LANG}.png")
plt.show()

In [ ]:
# ---- Evaluation: PPL / BPB, BLEU / chrF / ROUGE-L, diversity ------------
# Both held-out splits. Validation is what 2.3 asks for by name and what model
# selection already used; test is the untouched split, so it is the honest
# number to headline in the report -- best.pt was chosen on val, which makes the
# val figure very slightly optimistic. Test costs about a minute more.
import subprocess, sys

for split in ("val", "test"):
    subprocess.run([sys.executable, "-m", "common.modeling.evaluate",
                    "--lang", LANG,
                    "--checkpoint", str(out_dir / "best.pt"),
                    "--data-dir", DATA_DIR,
                    "--split", split,
                    "--out", f"{WORK}/{LANG}/eval/metrics_{split}.json"],
                   check=True)

In [ ]:
# ---- Attention analysis: heatmaps, entropy, mean distance ---------------
import subprocess, sys
subprocess.run([sys.executable, "-m", "common.modeling.attention",
                "--lang", LANG,
                "--checkpoint", str(out_dir / "best.pt"),
                "--data-dir", DATA_DIR,
                "--out-dir", f"{WORK}/{LANG}/eval"], check=True)

## After the run

`/kaggle/working/<lang>/` now holds:

| Path | What |
|---|---|
| `model/best.pt` | best-validation checkpoint — Phase 3 finetunes from this |
| `model/last.pt` | latest step, for resuming |
| `model/train_log.jsonl` | per-step losses; the loss curve is plotted from it |
| `model/loss_curve_<lang>.png` | loss curve figure |
| `model/parameter_report.json` | exact parameter count and breakdown |
| `model/config.json` | model + training hyperparameters, in plain text |
| `eval/metrics_val.json` | PPL, BPB, BLEU, chrF, ROUGE-L, diversity (validation) |
| `eval/metrics_test.json` | the same on the untouched test split |
| `eval/attention_heatmaps_<lang>.png` | query-vs-key heatmaps, early and late layer |
| `eval/attention_summary_<lang>.png` | per-head entropy and mean distance |
| `eval/attention_stats_<lang>.json` | the same numbers, plus the example passage |

**Commit the notebook** (*Save Version → Save & Run All*) so these persist.

Then, back in the repo: copy the logs, curves, metrics and figures into
`<lang>/model/` and `<lang>/eval/` and commit them (they are small and graded —
spec: *"training logs and loss curves in the repo"*). Upload the two `.pt`
checkpoints to Google Drive and put the links in the README — they are far too
large for git.